In [ ]:
import numpy as np
from scipy.integrate import solve_bvp
from scipy.interpolate import RBFInterpolator
import pandas as pd


# Set up w and dw for variable current

In [52]:
def rbf_gradient(interp, x):
    """
    Analytically compute gradient of a thin_plate_spline RBFInterpolator.
    x: (M, 2) array of query points
    returns: (M, 2) gradient
    """
    x = np.atleast_2d(x)
    x_norm = (x - interp._shift) / interp._scale          # (M, d)
    y_norm = (interp.y - interp._shift) / interp._scale   # (N, d)
    coeffs = interp._coeffs.ravel()                        # (N + p,)

    diff = x_norm[:, None, :] - y_norm[None, :, :]        # (M, N, d)
    r    = np.sqrt(np.sum(diff**2, axis=-1))               # (M, N)

    # d(phi)/d(r) = 2r*log(r) + r = r*(2*log(r)+1)
    # d(phi)/d(xj) = dphi_dr * (xj - yij) / r = (2*log(r)+1) * diff
    with np.errstate(divide='ignore', invalid='ignore'):
        factor = np.where(r == 0.0, 0.0, 2 * np.log(np.where(r == 0, 1, r)) + 1)  # (M, N)

    # kernel gradient: (M, N, d)
    kernel_grad = factor[:, :, None] * diff  # chain rule, r cancels

    # rbf contribution: sum over N, weighted by coeffs[:N]
    grad = np.einsum('mnd,n->md', kernel_grad, coeffs[:len(y_norm)])  # (M, d)

    # polynomial contribution: degree 1 -> [1, x0, x1], derivs are coeffs[N+1], coeffs[N+2]
    poly_coeffs = coeffs[len(y_norm):]                     # (p,)
    grad += poly_coeffs[1:] / interp._scale                # chain rule through normalization

    return grad

In [54]:
all_data = pd.read_csv('data/green_bay_wind_snapshot.csv')

X, Y, theta, speed = all_data['lon'], all_data['lat'], np.deg2rad(all_data['waveMeanDirection'].to_numpy()), 1 / all_data['waveTp'].to_numpy()

u, v = speed * np.sin(theta), speed * np.cos(theta)

# coordinates
y = np.column_stack([X, Y])

u_interp = RBFInterpolator(y, u, kernel="thin_plate_spline", smoothing=0.0)
v_interp = RBFInterpolator(y, v, kernel="thin_plate_spline", smoothing=0.0)


def w(s):
    return np.array([u_interp(s), v_interp(s)])


rbf_gradient(u_interp, np.array([-87.0, 45.0])[None, :]).shape

(1, 2)

In [60]:
def w(s):
    return np.array([u_interp(s), v_interp(s)]).T.squeeze(0)

def dw(s):
    return np.vstack([rbf_gradient(u_interp, s), rbf_gradient(v_interp, s)])

s = np.array([-87.0, 45.0])[None, :]

print(w(s))
print(dw(s))


[-0.50586324 -0.16915588]
[[-0.73542946  0.58859669]
 [-0.06641628 -0.81903718]]


In [ ]:
def w(s):
    return np.array([u_interp(s), v_interp(s)]).T.squeeze(0)

def dw(s):
    return np.vstack([rbf_gradient(u_interp, s), rbf_gradient(v_interp, s)])

def ode(t, Y, p):
    v = 10
    T = p[0]
    x, y, l1, l2 = Y
    lnorm = np.sqrt(l1**2 + l2**2) + 1e-10

    u1 = v * l1 / lnorm
    u2 = v * l2 / lnorm

    w1, w2 = w(x, y)
  
    dx = T * (u1 + w1)
    dy = T * (u2 + w2)

    Dw = dw(x, y)
    dl1, dl2 = -T * (Dw.T @ np.array([l1, l2]))

    return np.array([dx, dy, dl1, dl2])
    
x0 = np.array([0, 0])
xf = np.array([4, 4])
def bc(ya, yb, p):
    T = p[0]
    v=10

    xa, ya, l1a, l2a = ya
    xb, yb, l1b, l2b = yb

    lam_norm = np.sqrt(l1b**2 + l2b**2) + 1e-10
    u1b = v * l1b / lam_norm
    u2b = v * l2b / lam_norm
    wb = w(xb, yb).squeeze(1)
    
    Hf = l1b * (u1b + wb[0]) + l2b * (u2b + wb[1]) - 1.0

    return np.array([
        xa - x0[0],
        ya - x0[1],
        xb - xf[0],
        yb - xf[1],
        Hf
    ])


t = np.linspace(0, 1, 100)

# initial guess
Y_guess = np.zeros((4, t.size))
Y_guess[0] = np.linspace(x0[0], xf[0], t.size)
Y_guess[1] = np.linspace(x0[1], xf[1], t.size)
Y_guess[2] = 1.0
Y_guess[3] = 1.0

p_guess = np.array([2.0])

sol = solve_bvp(ode, bc, t, Y_guess, p=p_guess)